<a href="https://colab.research.google.com/github/adhishmodi/customer-churn-prediction/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Customer Churn Prediction — Data Preprocessing

Prepare the IBM Telco Customer Churn dataset for machine learning. This notebook uses the reusable functions in `src/preprocess.py`, preserves the raw CSV, and fits every learned transformation on the training split only.

**Important:** the transformed train/test arrays are useful for inspecting preprocessing. For cross-validation and model selection, use a complete scikit-learn pipeline on the raw training features so every fold fits its own transformations.


## 1. Runtime and dataset setup

Open this notebook in Google Colab, then upload the original CSV if it is not already present in `data/raw/`. The dataset is not committed to GitHub.

In [1]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/adhishmodi/customer-churn-prediction.git"
EXPECTED_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
current_dir = Path.cwd()
PROJECT_ROOT = next((p for p in (current_dir, *current_dir.parents) if (p / "src").is_dir() and (p / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/content/customer-churn-prediction")
    if not (PROJECT_ROOT / "src").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "raw" / EXPECTED_CSV
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")

Project root: /content/customer-churn-prediction
Dataset path: /content/customer-churn-prediction/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [2]:
if not DATA_PATH.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(f"Place {EXPECTED_CSV} in {DATA_PATH.parent}") from exc
    print(f"Upload the original file: {EXPECTED_CSV}")
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if EXPECTED_CSV in uploaded:
        selected_name = EXPECTED_CSV
    elif len(csv_names) == 1:
        selected_name = csv_names[0]
    else:
        raise ValueError(f"Upload exactly one CSV, preferably named {EXPECTED_CSV}. Found: {csv_names}")
    DATA_PATH.write_bytes(uploaded[selected_name])
assert DATA_PATH.exists(), "Dataset upload did not create the expected file."

Upload the original file: WA_Fn-UseC_-Telco-Customer-Churn.csv


Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv


## 2. Load and validate the raw data

Keep `raw_df` unchanged. All cleaning happens on a separate copy. Validation checks the expected schema, target, identifiers, missing values, and basic data quality.

In [3]:
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from src.data_loader import load_raw_data
from src.validate import validate_dataset, print_validation_results
from src.preprocess import (
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, clean_data,
    prepare_features_target, split_data, get_preprocessing_pipeline,
    transform_data, get_feature_names_out, save_preprocessor,
    save_processed_data, load_preprocessor,
)

raw_df = load_raw_data(DATA_PATH)
print(f"Raw shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
print(f"scikit-learn version: {sklearn.__version__}")
display(raw_df.head())
validation = validate_dataset(raw_df)
print_validation_results(validation)

Raw shape: 7,043 rows × 21 columns
scikit-learn version: 1.6.1


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


DATA VALIDATION RESULTS
Overall: PASSED

WARNINGS:
  [WARN] TotalCharges has 11 non-numeric values (likely blank strings). Conversion to numeric required.
  [WARN] High cardinality categorical column 'TotalCharges': 6531 unique values

INFO:
  [INFO] class_distribution: {'No': 5174, 'Yes': 1869}
  [INFO] minority_class_pct: 26.54
  [INFO] unique_ids: 7043
  [INFO] totalcharges_non_numeric: 11
  [INFO] SeniorCitizen_range: {'min': 0.0, 'max': 1.0}
  [INFO] tenure_range: {'min': 0.0, 'max': 72.0}
  [INFO] MonthlyCharges_range: {'min': 18.25, 'max': 118.75}
  [INFO] gender_cardinality: 2
  [INFO] Partner_cardinality: 2
  [INFO] Dependents_cardinality: 2
  [INFO] PhoneService_cardinality: 2
  [INFO] MultipleLines_cardinality: 3
  [INFO] InternetService_cardinality: 3
  [INFO] OnlineSecurity_cardinality: 3
  [INFO] OnlineBackup_cardinality: 3
  [INFO] DeviceProtection_cardinality: 3
  [INFO] TechSupport_cardinality: 3
  [INFO] StreamingTV_cardinality: 3
  [INFO] StreamingMovies_cardinality:

## 3. Deterministic cleaning

These operations do not estimate statistics from the dataset:
- Convert `TotalCharges` to numeric; non-numeric blanks become missing values.
- Set `TotalCharges` to 0 only when it is missing and `tenure` is 0 (the known no-tenure case).
- Convert `SeniorCitizen` values 0/1 to readable `No`/`Yes` categories.
- Remove `customerID`, which is an identifier rather than a predictive feature.

Other missing values are deliberately retained for imputation after the split.

In [4]:
cleaned_df = clean_data(raw_df)
print("Raw dataframe unchanged:", "customerID" in raw_df.columns and raw_df["TotalCharges"].dtype == "object")
print("Cleaned shape:", cleaned_df.shape)
print("Identifier removed:", "customerID" not in cleaned_df.columns)
print("SeniorCitizen values:", sorted(cleaned_df["SeniorCitizen"].dropna().unique().tolist()))
print("Remaining missing values:")
display(cleaned_df.isna().sum().loc[lambda values: values.gt(0)].to_frame("missing"))
display(cleaned_df[["tenure", "TotalCharges", "SeniorCitizen"]].head())

Raw dataframe unchanged: True
Cleaned shape: (7043, 20)
Identifier removed: True
SeniorCitizen values: ['No', 'Yes']
Remaining missing values:


,missing


,tenure,TotalCharges,SeniorCitizen
0,1,29.85,No
1,34,1889.50,No
2,2,108.15,No
3,45,1840.75,No
4,2,151.65,No


## 4. Separate features and target, then split

`Churn` is encoded as `No = 0` and `Yes = 1`. We use a stratified 80/20 split with random seed 42. Stratification keeps the churn proportion approximately consistent in both subsets.

**Leakage rule:** the split happens before fitting imputers, scalers, or encoders.

In [5]:
X, y = prepare_features_target(cleaned_df)
X_train, X_test, y_train, y_test = split_data(X, y)
print(f"Training rows: {len(X_train):,} ({len(X_train)/len(X):.1%})")
print(f"Test rows: {len(X_test):,} ({len(X_test)/len(X):.1%})")
display(pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "train_percent": y_train.value_counts(normalize=True).sort_index().mul(100).round(2),
    "test_count": y_test.value_counts().sort_index(),
    "test_percent": y_test.value_counts(normalize=True).sort_index().mul(100).round(2),
}).rename(index={0: "No churn (0)", 1: "Churn (1)"}))
print("Numeric features:", NUMERIC_FEATURES)
print("Categorical features:", CATEGORICAL_FEATURES)
print("Feature count before encoding:", X_train.shape[1])

Training rows: 5,634 (80.0%)
Test rows: 1,409 (20.0%)


,train_count,train_percent,test_count,test_percent
Churn,,,,
No churn (0),4139,73.46,1035,73.46
Churn (1),1495,26.54,374,26.54


Numeric features: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Feature count before encoding: 19


## 5. Build and fit the preprocessing pipeline

| Feature type | Missing-value strategy | Transformation | Reason |
|---|---|---|---|
| Numeric | Median imputation | Standard scaling | Median is less sensitive to extreme values; scaling helps distance- and gradient-based models. |
| Categorical | Most-frequent imputation | One-hot encoding | Converts categories into numeric indicator columns. |

`handle_unknown="ignore"` allows categories first seen during testing/inference without crashing. `drop="first"` removes one reference category per feature to avoid redundant dummy columns in linear models.

The preprocessor is fitted **only on `X_train`**. The test set is transformed with those already-learned statistics and categories.

In [6]:
preprocessor = get_preprocessing_pipeline()
preprocessor.fit(X_train)
X_train_processed = transform_data(preprocessor, X_train)
X_test_processed = transform_data(preprocessor, X_test)
feature_names = get_feature_names_out(preprocessor)
print("Training matrix:", X_train_processed.shape)
print("Test matrix:", X_test_processed.shape)
print("Transformed feature count:", len(feature_names))
print("All training values finite:", bool(np.isfinite(X_train_processed).all()))
print("All test values finite:", bool(np.isfinite(X_test_processed).all()))
display(pd.DataFrame({"feature": feature_names}).head(30))

Training matrix: (5634, 30)
Test matrix: (1409, 30)
Transformed feature count: 30
All training values finite: True
All test values finite: True


,feature
0,tenure
1,MonthlyCharges
2,TotalCharges
3,gender_Male
4,SeniorCitizen_Yes
5,Partner_Yes
6,Dependents_Yes
7,PhoneService_Yes
8,MultipleLines_No phone service
9,MultipleLines_Yes


## 6. Inspect what the transformer learned

The numeric imputer's statistics and categorical encoder's categories are learned from the training subset. The test data does not participate in fitting.

In [7]:
numeric_imputer = preprocessor.named_transformers_["numeric"].named_steps["imputer"]
categorical_encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]
display(pd.DataFrame({"numeric_feature": NUMERIC_FEATURES, "training_median": numeric_imputer.statistics_}))
for column, categories in zip(CATEGORICAL_FEATURES, categorical_encoder.categories_):
    print(f"{column}: {list(categories)}")

,numeric_feature,training_median
0,tenure,29.000
1,MonthlyCharges,70.500
2,TotalCharges,1394.925


gender: ['Female', 'Male']
SeniorCitizen: ['No', 'Yes']
Partner: ['No', 'Yes']
Dependents: ['No', 'Yes']
PhoneService: ['No', 'Yes']
MultipleLines: ['No', 'No phone service', 'Yes']
InternetService: ['DSL', 'Fiber optic', 'No']
OnlineSecurity: ['No', 'No internet service', 'Yes']
OnlineBackup: ['No', 'No internet service', 'Yes']
DeviceProtection: ['No', 'No internet service', 'Yes']
TechSupport: ['No', 'No internet service', 'Yes']
StreamingTV: ['No', 'No internet service', 'Yes']
StreamingMovies: ['No', 'No internet service', 'Yes']
Contract: ['Month-to-month', 'One year', 'Two year']
PaperlessBilling: ['No', 'Yes']
PaymentMethod: ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']


## 7. Save the fitted preprocessing artifacts

Artifacts are written to `data/processed/`, which is intentionally ignored by Git. The raw CSV remains untouched. The fitted transformer must be reused for future inference data; do not fit a new transformer on individual prediction batches.

In [8]:
from src.config import PROCESSED_DATA_DIR
preprocessor_path = PROCESSED_DATA_DIR / "preprocessor.joblib"
save_preprocessor(preprocessor, preprocessor_path)
save_processed_data(
    X_train_processed, X_test_processed, y_train, y_test,
    feature_names, output_dir=PROCESSED_DATA_DIR,
)
reloaded_preprocessor = load_preprocessor(preprocessor_path)
reloaded_test = transform_data(reloaded_preprocessor, X_test)
assert np.allclose(reloaded_test, X_test_processed)
print(f"Saved preprocessing artifacts to: {PROCESSED_DATA_DIR}")
print("Reload check passed:", reloaded_test.shape == X_test_processed.shape)

Saved preprocessing artifacts to: /content/customer-churn-prediction/data/processed
Reload check passed: True


## 8. Preprocessing checks and conclusion

Before moving to model training, verify that the raw data is unchanged, the identifier and target are not included in `X`, the split is stratified, all transformed values are finite, and the same fitted transformer can process the test set. The following checks summarize the run.

In [9]:
assert "customerID" not in X_train.columns
assert "Churn" not in X_train.columns
assert len(X_train) + len(X_test) == len(X)
assert X_train_processed.shape[1] == X_test_processed.shape[1] == len(feature_names)
assert np.isfinite(X_train_processed).all() and np.isfinite(X_test_processed).all()
assert raw_df["TotalCharges"].dtype == "object"
print("All preprocessing assertions passed.")
print(f"Rows: {len(X):,} | Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Input features: {X_train.shape[1]} | Encoded features: {len(feature_names)}")
print(f"Train churn rate: {y_train.mean():.2%} | Test churn rate: {y_test.mean():.2%}")
print("Next milestone: model training with cross-validation on raw X_train and y_train.")

All preprocessing assertions passed.
Rows: 7,043 | Train: 5,634 | Test: 1,409
Input features: 19 | Encoded features: 30
Train churn rate: 26.54% | Test churn rate: 26.54%
Next milestone: model training with cross-validation on raw X_train and y_train.


### Key leakage reminder

These saved transformed arrays are a preprocessing snapshot for inspection and reproducibility. For cross-validation, do **not** cross-validate on `X_train_processed`. Instead, pass raw `X_train` into a complete pipeline that contains both `get_preprocessing_pipeline()` and the classifier. That makes each fold learn its own imputation, scaling, and encoding from that fold's training rows only.